# End-to-end pipeline demonstration — record §96

One fundus image in; out come the quality check (Stage 1), the preprocessed frame (Stage 2), the vessel map (Stage 3),
the lesion maps (Stage 4), the grade from the three-seed joint grader with its class probabilities, the ensemble
uncertainty, and a Grad-CAM map of what raised the grade. Every model is frozen; nothing is trained.

- The grade comes from the RGB grader alone. The vessel and lesion maps are supporting views.
- **Research prototype — not a medical device; its output is not a diagnosis.**
- Use images the project may show: APTOS validation images, or your own. Do not use DDR or IDRiD test images for the demo.

Runs on a T4 in about a minute per image after the models are loaded (the Stage-4 lesion model runs at 1536 px).

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. 
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad
import shutil, tarfile
import ddr_grading_data as dd, ddr_grading_eval as ev
import joint_train as jt, final_external_eval as fx, uncertainty_eval as ue, gradcam, demo_pipeline as demo
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] WHICH MODEL AND WHICH IMAGES ====
ARM = "direct"            # the pre-declared primary grader: joint APTOS + EyePACS on direct frames ("crop" shows the ablation)
UNCERTAINTY_THRESHOLD = None   # an entropy value from the Stage-10 validation results; None reports the scores without a flag
N_IMAGES = 6              # how many APTOS validation images to show (one or two per grade, in validation order)
EXTRA_IMAGES = []         # paths of your own images, if any
print({"ARM": ARM, "N_IMAGES": N_IMAGES, "UNCERTAINTY_THRESHOLD": UNCERTAINTY_THRESHOLD})

In [ ]:
# ==== [3] LOAD THE FROZEN MODELS (once) ====
DRIVE_ROOT = colab_config.DRIVE.project_root
EXP = colab_config.DRIVE.experiments_root
PARENT = fx.ig.load_protocol()[0]
_root = lambda rel: posixpath.join(DRIVE_ROOT, *rel.split("/"))
CHECKPOINTS = {}
for _s in jt.SEEDS:
    _run = jt.run_dir_for(EXP, _s, ARM)
    _best = json.load(open(posixpath.join(_run, "result.json")))["best"]["checkpoint"]
    _path = posixpath.join(_run, "checkpoints", _best["slot"], "model.weights.h5")
    assert fx.ig.sha256_file(_path) == _best["weights_sha256"], f"seed {_s}: not the evaluated checkpoint"
    CHECKPOINTS[_s] = _path
PRETRAINED = pl.ensure_pretrained_weights(posixpath.join(EXP, "PL_ConvNeXtPriors", "pretrained_weights"))["local_path"]
PIPELINE = demo.Pipeline(CHECKPOINTS, PRETRAINED, _root(PARENT["checkpoints"]["stage3_lwnet"]["path"]), _root(PARENT["checkpoints"]["stage4_unet"]["path"]),
                         iqa_path=posixpath.join(colab_config.EXPORTED_MODELS_ROOT, "IQA", "best_model.keras"), arm=ARM,
                         uncertainty_threshold=UNCERTAINTY_THRESHOLD)
print("loaded:", {"grader seeds": PIPELINE.seeds, "arm": ARM, "device": PIPELINE.settings["device"]})

In [ ]:
# ==== [4] RUN ON THE IMAGES AND SHOW THE RESULT ====
import pathology_grader_severity_fusion as sf
from IPython.display import Image as _Image, display
_ids, _grades = sf.authoritative_validation_ids()
_pick = fx.ig.parity_subset(_ids, _grades, max(1, N_IMAGES // 5))[:N_IMAGES]              # the first images of every grade, in validation order
PATHS = [posixpath.join(colab_config.APTOS2019_RAW_DIR, "train_images", f"{i}.png") for i in _pick] + list(EXTRA_IMAGES)
OUT_DIR = posixpath.join(EXP, "FinalExternal", f"demo_{ARM}")
ROWS = demo.run_folder(PIPELINE, PATHS, OUT_DIR)
display(pd.DataFrame(ROWS))
for _p in PATHS:
    display(_Image(posixpath.join(OUT_DIR, os.path.splitext(os.path.basename(_p))[0] + ".png")))